# 가설 검정 · 시각화
## 법률 서비스 접근성 불균형 분석 (2/2)

서강대 26-1 빅데이터컴퓨팅 · 20210717 김찬우

`01_분석정리.ipynb`에서 산출한 LLAI·클러스터 결과를 바탕으로 가설을 검정하고 시각화한다.  
검정·도표 로직은 `src/model/hypothesis.py`, `src/viz.py`에 있다.

## 가설

| 가설 | 내용 | 검정 방법 | 데이터 제약 |
|---|---|---|---|
| H1 | 수도권 변호사 1인당 인구 < 비수도권 | Mann-Whitney U | 표본 작음(3 vs 10) |
| H2 | 변호사 적을수록 사건부담↑ | corr(A1, A2) | — |
| H3 | 소득 낮을수록 법률구조 수요↑·자원↓ | corr(GRDP, A3/A1) | GRDP 2022 단면 |
| H4 | 로스쿨 이후 격차 미축소 | 법률구조 격차(변동계수) 추세 | 변호사 시계열 부재 → A3로 대체 |

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from model import hypothesis
import viz
viz.setup_font()  # 한글 폰트(Malgun Gothic)
%matplotlib inline
print('준비 완료')

## 1. 가설 검정 결과

In [ ]:
hypothesis.run('region13')   # 부산·인천·울산 분리 단위

In [ ]:
hypothesis.run('region10')   # 공단 기준 10도단위

In [ ]:
hypothesis.h4_trend()        # H4 보조: 법률구조 격차 추세

### 검정 해석
- **H2 — 강하게 지지.** corr(A1, A2) = −0.84~−0.88 (p<0.01). 변호사가 많은 권역일수록 1인당 사건부담이 낮다.
- **H1 — 방향은 일치, 통계적으로는 불충분.** 수도권 변호사 1인당 인구(중앙값 4,039명) < 비수도권(5,250명)이나 표본이 작아(n=3 vs 10) p>0.05.
- **H3 — 약함.** GRDP와 변호사 접근성(A1)은 약한 양의 상관(소득 높은 곳에 변호사 많음 경향), 법률구조(A3)와는 일관되지 않음.
- **H4 — 격차 미축소 지지.** 법률구조 권역 격차(변동계수)가 2012년 0.85 → 2025년 0.90으로 오히려 확대.

## 2. 시각화

PNG로 저장하려면 터미널에서 `python src/viz.py` 실행 → `outputs/figures/`.

In [ ]:
# LLAI 순위 (클러스터 색)
viz.fig_llai_ranking('region13', save=False)

In [ ]:
# 변호사 접근성(A1) vs 사건부담(A2) — H2
viz.fig_scatter_a1_a2('region13', save=False)

In [ ]:
# 정규화 세부지표 히트맵 (A1/A2/A3)
viz.fig_subindicators('region13', save=False)

In [ ]:
# 가중치 3종별 LLAI 비교 (강건성)
viz.fig_weights('region13', save=False)

In [ ]:
# H1: 수도권 vs 비수도권 변호사 1인당 인구
viz.fig_h1_box(save=False)

In [ ]:
# H4: 법률구조 권역 격차 추세
viz.fig_h4_trend(save=False)

## 3. 종합 결론

1. **변호사 자원의 서울 일극 집중**이 법률 접근성 격차의 핵심 동인. 서울은 모든 지표에서 압도적이며 단독 클러스터로 분리된다.
2. **변호사 수와 사건부담은 강한 음의 관계(H2)** — 변호사가 부족한 비수도권일수록 1인당 업무 부담이 커져 실질 서비스 질 저하로 이어질 수 있다.
3. **격차는 축소되지 않고 있다(H4)** — 로스쿨 이후 변호사 총량 증가에도 법률구조 격차는 확대 추세.
4. **정책 시사점** — 지방 변호사 유치 인센티브, 법률구조공단 자원의 비수도권 우선 배치, 사건부담 높은 권역 인력 보강.

### 한계와 향후 과제
- 변호사 데이터가 현재 스냅샷이라 LLAI 시계열·H4 직접 검정 불가 → 변호사백서 PDF로 과거 시계열 보강 필요.
- 표본(권역 10~13개)이 작아 통계적 검정력 제한.
- 코로플레스 지도(folium/geopandas)는 행정구역 경계 데이터 확보 후 추가 예정.